# 07 - ML Data Prep: Forms-Only + Nested Instructions -> JSONL

**NewStart AI - Document Routing (all-in-notebook edition)**

Self-contained and PEP 8 compliant: every function is defined inline, no local
`.py` imports, lines wrapped to 79 columns. Turns
`data/processed/final_dataset.csv` into a model-ready dataset for **agency**
routing (label in {USCIS, DMV, SSA, IRS}) by (1) keeping forms only,
(2) nesting each form's instructions inside it, and (3) writing leakage-safe,
class-balanced JSONL splits.


## 1. Setup, imports, and paths

In [1]:
try:
    import google.colab  # noqa: F401
    !pip -q install scikit-learn pandas
except Exception:
    pass

In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

LABEL_COL = "agency"
MIN_TEXT_LEN = 50
RANDOM_STATE = 42


def find_base():
    """Return the project root (Colab Drive or a local path)."""
    candidates = [
        Path("/content/drive/MyDrive/newstart_ai"),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for root in candidates:
        target = root / "data" / "processed" / "final_dataset.csv"
        if target.exists():
            return root
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/newstart_ai")
    except Exception as exc:
        raise FileNotFoundError(
            "Could not locate data/processed/final_dataset.csv"
        ) from exc


BASE_DIR = find_base()
IN_CSV = BASE_DIR / "data" / "processed" / "final_dataset.csv"
OUT_DIR = BASE_DIR / "data" / "ml"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

BASE_DIR: /sessions/magical-nice-archimedes/mnt/FinalSchoolProject/newstart-ai-dev-petra/newstart-ai-dev-petra/newstart_ai_jeffi


## 2. Load and basic cleaning

In [3]:
df = pd.read_csv(IN_CSV)
df["text"] = df["text"].astype(str)
df["text_length"] = df["text"].str.len()

before = len(df)
keep = (df["text_length"] >= MIN_TEXT_LEN) & df[LABEL_COL].notna()
df = df[keep].reset_index(drop=True)
print(f"Loaded {before} rows, kept {len(df)} after cleaning")
print(df["document_type"].value_counts())

Loaded 754 rows, kept 754 after cleaning
document_type
form               587
instructions       144
supplement          14
checklist            6
translated_form      3
Name: count, dtype: int64


## 3. Document-family key (links a form to its instructions)

The key is the normalized ``form_number`` when present, otherwise a code parsed
from the filename, prefixed by the agency to avoid cross-agency collisions.


In [4]:
def norm_form(form_number):
    """Normalize a form number to lowercase alphanumerics, else None."""
    is_nan = isinstance(form_number, float) and pd.isna(form_number)
    if form_number is None or is_nan:
        return None
    value = str(form_number).strip()
    if not value:
        return None
    return re.sub(r"[^a-z0-9]", "", value.lower())


CODE_RE = re.compile(r"([a-z]{2,4})[-_ ]?(\d{2,5})")
SUFFIX_RE = re.compile(
    r"(instructions|instr|worksheet|supplement|checklist"
    r"|translated|508c|508|sp|rev|\d{4})"
)


def family_from_filename(name):
    """Derive a family key from a filename when no form number exists."""
    stem = str(name).lower().rsplit(".", 1)[0]
    codes = CODE_RE.findall(stem)
    if codes:
        prefix, number = codes[-1]
        return prefix + number
    stem = SUFFIX_RE.sub("", stem)
    return re.sub(r"[^a-z0-9]", "", stem) or str(name).lower()


def make_family(row):
    """Build the agency-prefixed family key for a dataframe row."""
    base = norm_form(row["form_number"])
    if not base:
        base = family_from_filename(row["filename"])
    return f"{str(row[LABEL_COL]).lower()}:{base}"


df["family"] = df.apply(make_family, axis=1)
print("families:", df["family"].nunique(), "over", len(df), "docs")

families: 538 over 754 docs


## 4. Restrict to forms and nest the associated instructions

Primary records are ``document_type == "form"``. The primary form of a family is
the form with the longest text; every non-form doc in that family is nested into
it. Non-form docs whose family has no form are orphans and are dropped.


In [5]:
forms = df[df["document_type"] == "form"].copy()
nonform = df[df["document_type"] != "form"].copy()

# Primary form per family = the form with the longest text.
primary_ids = set(
    forms.loc[
        forms.groupby("family")["text_length"].idxmax(),
        "document_id",
    ]
)
fam_to_nonform = dict(list(nonform.groupby("family")))


def build_associated(family):
    """Return nested instruction dicts for a family, if any."""
    associated = []
    group = fam_to_nonform.get(family)
    if group is None:
        return associated
    for _, doc in group.iterrows():
        associated.append({
            "filename": doc["filename"],
            "document_type": doc["document_type"],
            "text": doc["text"],
            "page_count": int(doc["page_count"]),
            "text_length": int(doc["text_length"]),
        })
    return associated


def build_record(form):
    """Build one nested form record with its associated instructions."""
    associated = []
    if form["document_id"] in primary_ids:
        associated = build_associated(form["family"])
    form_number = form["form_number"]
    return {
        "document_id": int(form["document_id"]),
        "filename": form["filename"],
        "agency": form["agency"],
        "form_number": None if pd.isna(form_number) else form_number,
        "document_type": "form",
        "family": form["family"],
        "text": form["text"],
        "page_count": int(form["page_count"]),
        "text_length": int(form["text_length"]),
        "associated_instructions": associated,
        "n_associated": len(associated),
    }


records = pd.DataFrame([build_record(f) for _, f in forms.iterrows()])
orphans = nonform[~nonform["family"].isin(set(forms["family"]))]
with_instr = int((records["n_associated"] > 0).sum())
print("primary FORM records:", len(records))
print("non-form docs nested:", int(records["n_associated"].sum()))
print("forms with >=1 instruction:", with_instr)
print("orphan non-form docs dropped:", len(orphans))
print(records["agency"].value_counts())

primary FORM records: 587
non-form docs nested: 139
forms with >=1 instruction: 79
orphan non-form docs dropped: 28
agency
DMV      277
SSA      173
USCIS    123
IRS       14
Name: count, dtype: int64


## 5. Leakage-safe, stratified split (grouped by family)

``StratifiedGroupKFold`` keeps each family in a single split while stratifying by
agency. A hard assertion verifies zero family overlap.


In [6]:
y = records[LABEL_COL].values
groups = records["family"].values

outer = StratifiedGroupKFold(
    n_splits=7, shuffle=True, random_state=RANDOM_STATE
)
trainval_idx, test_idx = next(outer.split(records, y, groups))
trainval = records.iloc[trainval_idx].reset_index(drop=True)
test = records.iloc[test_idx].reset_index(drop=True)

inner = StratifiedGroupKFold(
    n_splits=6, shuffle=True, random_state=RANDOM_STATE
)
tr_idx, val_idx = next(
    inner.split(trainval, trainval[LABEL_COL], trainval["family"])
)
train = trainval.iloc[tr_idx].reset_index(drop=True)
val = trainval.iloc[val_idx].reset_index(drop=True)

for name, part in [("train", train), ("val", val), ("test", test)]:
    counts = dict(part[LABEL_COL].value_counts())
    print(f"{name:5s} n={len(part):3d}  {counts}")

fam_train = set(train["family"])
fam_val = set(val["family"])
fam_test = set(test["family"])
overlaps = {
    "train/val": len(fam_train & fam_val),
    "train/test": len(fam_train & fam_test),
    "val/test": len(fam_val & fam_test),
}
assert sum(overlaps.values()) == 0, f"LEAKAGE: {overlaps}"
print("PASS - no family crosses splits:", overlaps)

train n=416  {'DMV': np.int64(194), 'SSA': np.int64(125), 'USCIS': np.int64(86), 'IRS': np.int64(11)}
val   n= 89  {'DMV': np.int64(41), 'SSA': np.int64(24), 'USCIS': np.int64(23), 'IRS': np.int64(1)}
test  n= 82  {'DMV': np.int64(42), 'SSA': np.int64(24), 'USCIS': np.int64(14), 'IRS': np.int64(2)}
PASS - no family crosses splits: {'train/val': 0, 'train/test': 0, 'val/test': 0}


## 6. Class weights (imbalance-aware)

In [7]:
classes = np.array(sorted(records[LABEL_COL].unique()))
weights = compute_class_weight(
    "balanced", classes=classes, y=train[LABEL_COL]
)
class_weights = {
    cls: round(float(weight), 4)
    for cls, weight in zip(classes, weights)
}
label_map = {cls: index for index, cls in enumerate(classes)}
print("label_map:", label_map)
print("class_weights:", class_weights)

label_map: {np.str_('DMV'): 0, np.str_('IRS'): 1, np.str_('SSA'): 2, np.str_('USCIS'): 3}
class_weights: {np.str_('DMV'): 0.5361, np.str_('IRS'): 9.4545, np.str_('SSA'): 0.832, np.str_('USCIS'): 1.2093}


## 7. Write nested JSONL + artifacts

In [8]:
def write_jsonl(frame, path):
    """Write one JSON object per line, adding an integer label."""
    with open(path, "w", encoding="utf-8") as handle:
        for _, row in frame.iterrows():
            record = row.to_dict()
            record["label"] = int(label_map[row[LABEL_COL]])
            handle.write(json.dumps(record, ensure_ascii=False))
            handle.write("\n")
    return path


def split_summary(part):
    """Return counts + per-agency breakdown for a split."""
    by_agency = {
        key: int(value)
        for key, value in part[LABEL_COL].value_counts().items()
    }
    return {"n": int(len(part)), "by_agency": by_agency}


for name, part in [("train", train), ("val", val), ("test", test)]:
    write_jsonl(part, OUT_DIR / f"{name}.jsonl")

(OUT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2))

class_weights_payload = {
    "by_name": class_weights,
    "by_id": {
        str(label_map[cls]): class_weights[cls] for cls in classes
    },
}
(OUT_DIR / "class_weights.json").write_text(
    json.dumps(class_weights_payload, indent=2)
)

split_stats = {
    "n_form_records": int(len(records)),
    "n_nested_instructions": int(records["n_associated"].sum()),
    "forms_with_instructions": with_instr,
    "label_map": label_map,
    "leakage_family_overlaps": overlaps,
    "splits": {
        "train": split_summary(train),
        "val": split_summary(val),
        "test": split_summary(test),
    },
    "class_weights": class_weights,
}
(OUT_DIR / "split_stats.json").write_text(
    json.dumps(split_stats, indent=2)
)
print("wrote JSONL + artifacts to", OUT_DIR)

wrote JSONL + artifacts to /sessions/magical-nice-archimedes/mnt/FinalSchoolProject/newstart-ai-dev-petra/newstart-ai-dev-petra/newstart_ai_jeffi/data/ml


### 7b. Peek at one nested record

In [9]:
with open(OUT_DIR / "train.jsonl", encoding="utf-8") as handle:
    sample = json.loads(handle.readline())

print("agency:", sample["agency"], "| form:", sample["filename"])
print("associated_instructions:", sample["n_associated"])
for item in sample["associated_instructions"][:3]:
    print("  -", item["filename"], f"({item['document_type']})")

agency: USCIS | form: i-765ws.pdf
associated_instructions: 0


## 8. Summary & inferences

- Forms-only + nesting yields 587 form records; instructions are nested inside
  their parent form, which removes form-vs-instruction leakage by construction.
- We still group-split by family to guard multi-form variants.
- The forms-only restriction makes IRS very rare (~14 forms), so use macro-F1 and
  watch the IRS confusion row.

Next: ``08_train_baseline`` and ``09_train_distilbert`` read these JSONL files.
